# Lab 4: The what-if machine
*Python for Healthcare · Stanford Technology Training (ITS-2519) · Dr. Lionel Levine*
**Session 2 · 50 minutes: about 40 minutes of work, then we debrief together**

Kestrel Valley Health's birth center (fictional) delivers about 3,500 babies a year. The chief nursing officer
asks: *"We can fund one change next year: an eighth labor and delivery room, or a discharge-planning program that
shortens the average postpartum stay by 4 hours. Which should we pick?"*

You'll answer with a discrete-event simulation, then build a what-if dashboard she could use herself.

**The unit.** Patients arrive at triage (OBS), move to a labor, delivery and recovery room (LDR), then to a
postpartum bed (PP). There's no data file: the model's inputs are the numbers in `BASELINE`.

## Step 0. Install SimPy and load the model (1 minute)

In [ ]:
%pip install -q simpy

Colab doesn't include SimPy, so the cell above installs it (about 5 seconds). If it prints a note about restarting
the kernel, ignore it. Now run the model cell. It only defines things; nothing runs yet.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import simpy
from scipy import stats

# Kestrel Valley Health birth center (fictional). The structure follows Mark Isken's teaching
# model of an obstetrics unit: triage (OBS) -> labor, delivery and recovery (LDR) -> postpartum (PP).
# Times are in hours.
BASELINE = {
    "arrivals_per_day": 9.6,   # about 3,500 births a year
    "obs_beds": 4,             # triage beds
    "ldr_rooms": 7,
    "pp_beds": 26,
    "obs_hours": 3,            # average stay in each unit
    "ldr_hours": 12,
    "pp_hours": 48,
}
WARMUP_DAYS = 30               # the unit starts empty, so ignore the first month
RUN_DAYS = 365
MAX_TRIAGE_WAIT_H = 12         # past this average wait, a real unit would divert patients


def check_capacity(params):
    """Refuse to run a unit whose average demand meets or exceeds its beds: its queue never settles."""
    per_hour = params["arrivals_per_day"] / 24
    for unit, beds in [("obs", "obs_beds"), ("ldr", "ldr_rooms"), ("pp", "pp_beds")]:
        demand = per_hour * params[f"{unit}_hours"]          # beds busy on average
        if demand >= params[beds]:
            raise ValueError(f"Refusing to run: {unit.upper()} needs {demand:.1f} beds on average but has "
                             f"{params[beds]}. The queue would grow without limit.")


def simulate_year(params, seed):
    """Simulate one year of the birth center and return one row of results."""
    check_capacity(params)
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    obs = simpy.Resource(env, capacity=params["obs_beds"])
    ldr = simpy.Resource(env, capacity=params["ldr_rooms"])
    pp = simpy.Resource(env, capacity=params["pp_beds"])
    patients = []

    def patient(stays):
        arrived = env.now
        obs_bed = obs.request()
        yield obs_bed                          # wait for a triage bed
        wait_obs = env.now - arrived
        yield env.timeout(stays["obs"])

        ready = env.now
        ldr_room = ldr.request()
        yield ldr_room                         # wait for an LDR room, still holding the triage bed
        obs.release(obs_bed)
        wait_ldr = env.now - ready
        yield env.timeout(stays["ldr"])

        ready = env.now
        pp_bed = pp.request()
        yield pp_bed                           # blocked: wait for a PP bed, still holding the LDR room
        ldr.release(ldr_room)
        blocked = env.now - ready
        yield env.timeout(stays["pp"])
        pp.release(pp_bed)

        if arrived >= WARMUP_DAYS * 24:        # count only patients who arrive after the warm-up
            patients.append((wait_obs, wait_ldr, blocked))

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(24 / params["arrivals_per_day"]))
            if env.now > (WARMUP_DAYS + RUN_DAYS) * 24:
                return                         # stop admitting, and let everyone already here finish
            # Each patient's stays are drawn when she arrives, so two scenarios run with
            # the same seed see exactly the same patients.
            stays = {unit: rng.exponential(1.0) * params[f"{unit}_hours"] for unit in ("obs", "ldr", "pp")}
            env.process(patient(stays))

    env.process(arrivals())
    env.run()                                  # until the last patient goes home

    wait_obs, wait_ldr, blocked = np.array(patients).T
    return {
        "births counted": len(patients),
        "waited for a triage bed (%)": 100 * (wait_obs > 0).mean(),
        "mean wait for a triage bed, all patients (h)": wait_obs.mean(),
        "waited for an LDR room (%)": 100 * (wait_ldr > 0).mean(),
        "mean wait for an LDR room, all patients (h)": wait_ldr.mean(),
        "blocked in LDR, waiting for PP (%)": 100 * (blocked > 0).mean(),
    }


def run_many(params, n=30, first_seed=1):
    """Run n replications (one per seed) and return one row per replication."""
    return pd.DataFrame([simulate_year(params, seed) for seed in range(first_seed, first_seed + n)])


def summarize(results, min_reps=20):
    """Mean and 95% interval for each result. Refuses to report numbers that don't mean anything."""
    n = len(results)
    if n < min_reps:
        raise ValueError(f"Refusing to summarize {n} replication(s): run at least {min_reps}. "
                         "One run is an anecdote.")
    triage_wait = results["mean wait for a triage bed, all patients (h)"].mean()
    if triage_wait > MAX_TRIAGE_WAIT_H:
        raise ValueError(f"Refusing to summarize: patients waited {triage_wait:.0f} hours for a triage bed on "
                         "average. A real unit would divert patients long before that, and this model can't.")
    t = stats.t.ppf(0.975, n - 1)
    mean = results.mean()
    half = t * results.std() / np.sqrt(n)
    return pd.DataFrame({"mean": mean.round(1),
                         "95% interval": [f"{m - h:.1f} to {m + h:.1f}" for m, h in zip(mean, half)]})

## Step 1. Read the model (5 minutes)
```
arrive ──▶ triage (OBS) ──▶ labor & delivery (LDR) ──▶ postpartum (PP) ──▶ home
           4 beds, ~3 h      7 rooms, ~12 h            26 beds, ~48 h
```
Find the lines that answer these. Gemini's Learn Mode can help: *Explain `patient()` line by line.*
1. Where does a patient wait when every LDR room is full?
2. What does *blocked* mean in this model?
3. Why are the first 30 days ignored?
4. Which line makes two scenarios run with the same seed see exactly the same patients?

**Predict before you run anything.** On average the LDR rooms are busy about 70% of the time
(9.6 births a day × 12 hours ÷ 24 hours ÷ 7 rooms). What share of patients do you think wait for an LDR room?
Write down a number.

## Step 2. One simulated year (4 minutes)
Simulate the same unit for one year, twice, with two different random seeds.

In [ ]:
two_runs = pd.DataFrame({f"seed {seed}": simulate_year(BASELINE, seed) for seed in [1, 2]})
two_runs.round(1)

Same unit, same rules, same average demand. How different are the two years? If you had run only one, which
number would you have reported to the CNO?

## Step 3. Replications (8 minutes)
One run is an anecdote. Run 30 simulated years and report the average with a 95% interval
(about 5 seconds).

In [ ]:
results = run_many(BASELINE, n=30)        # 30 simulated years, seeds 1 to 30
summarize(results)

How much do single years vary?

In [ ]:
col = "blocked in LDR, waiting for PP (%)"
results[col].plot.hist(bins=12, figsize=(6, 3.5), title="Blocked in LDR: 30 simulated years, same unit")
plt.xlabel("Share of patients blocked in LDR (%)")
plt.ylabel("Simulated years")
plt.show()
print(f"Lowest year {results[col].min():.1f}%, highest year {results[col].max():.1f}%")

`summarize()` has a rule written into it. Run this cell and read the message.

In [ ]:
summarize(run_many(BASELINE, n=3))

1. How close was your prediction from Step 1?
2. What does the 95% interval describe: next year, or the long-run average? (Compare it with the histogram.)

## Step 4. Compare the two proposals (9 minutes)
**Before you run it, decide which number you'll use to choose, and write it down:**
the share who wait for an LDR room, the average wait, or the share blocked in LDR?

My deciding metric (double-click to type): ______

`compare()` runs the baseline and each proposal with the same 30 seeds, so each scenario sees the same patients.
It reports each proposal's *change* from baseline (10 to 20 seconds).

In [ ]:
def compare(proposals, n=30):
    """Run the baseline and each proposal on the same seeds, so each one sees the same patients.
    Show the baseline mean and each proposal's change from it, with a 95% interval."""
    base = run_many(BASELINE, n)
    t = stats.t.ppf(0.975, n - 1)
    table = pd.DataFrame({"baseline": base.mean().round(1)})
    for name, changes in proposals.items():
        diff = run_many({**BASELINE, **changes}, n) - base      # paired: same seed, same patients
        half = t * diff.std() / np.sqrt(n)
        table[name] = [f"{m:+.1f} ({m - h:+.1f} to {m + h:+.1f})" for m, h in zip(diff.mean(), half)]
    return table

compare({"8 LDR rooms": {"ldr_rooms": 8}, "PP stay 44 h": {"pp_hours": 44}})

1. Which proposal wins on your metric? Would it win on the others?
2. What happens to blocking with an eighth LDR room? Why?
3. Why is the change in births exactly zero?

## Step 5. Ask Gemini for a dashboard (10 minutes)
The CNO wants to try her own what-ifs. Send Gemini this request, then insert and run the code it writes:

`Write an ipywidgets dashboard for the birth-center model in this notebook. Use the existing BASELINE, run_many, and summarize; don't change them. Add sliders for triage beds (2 to 6), LDR rooms (5 to 10), PP beds (20 to 32), average PP stay in hours (36 to 56, step 2), and births per day (7 to 12, step 0.2), each starting at its baseline value, in a dictionary named sliders keyed by the BASELINE names. Add a slider named reps for replications (20 to 50, step 10, starting at 30), and a Run button whose click handler is named on_run. When clicked, run run_many with the slider values and show summarize() of the result. If the model refuses to run, show its message instead.`

Notice how much the request specifies: which functions to reuse, every slider's range and starting value, and what
to do when the model refuses. Each detail you leave out is a decision Gemini makes for you.

In [ ]:
# Dashboard: insert and run the code Gemini writes for "Write an ipywidgets dashboard..."

In [ ]:
#@title Fallback: Dashboard (only if Gemini isn't available: tick the box, then run this cell)
run_fallback = False  #@param {type:"boolean"}
if run_fallback:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    sliders = {
        "obs_beds": widgets.IntSlider(value=BASELINE["obs_beds"], min=2, max=6, description="Triage beds"),
        "ldr_rooms": widgets.IntSlider(value=BASELINE["ldr_rooms"], min=5, max=10, description="LDR rooms"),
        "pp_beds": widgets.IntSlider(value=BASELINE["pp_beds"], min=20, max=32, description="PP beds"),
        "pp_hours": widgets.IntSlider(value=BASELINE["pp_hours"], min=36, max=56, step=2, description="PP stay (h)"),
        "arrivals_per_day": widgets.FloatSlider(value=BASELINE["arrivals_per_day"], min=7, max=12, step=0.2,
                                                description="Births/day"),
    }
    reps = widgets.IntSlider(value=30, min=20, max=50, step=10, description="Replications")
    run_button = widgets.Button(description="Run", button_style="primary")
    output = widgets.Output()

    def on_run(_):
        params = {**BASELINE, **{name: round(s.value, 1) for name, s in sliders.items()}}   # 9.6, not 9.600000000000001
        with output:
            clear_output()
            print(f"Running {reps.value} simulated years...")
            try:
                table = summarize(run_many(params, n=reps.value))
            except ValueError as refusal:
                clear_output()
                print(refusal)
                return
            clear_output()
            display(table)

    run_button.on_click(on_run)
    display(widgets.VBox([*sliders.values(), reps, run_button, output]))

**Check it before you trust it.** Leave every slider at its starting value, keep 30 replications, and click
**Run**. The table must match your Step 3 table exactly. Then try 12 births a day with 22 PP beds.

If clicking **Run** shows nothing, run the next cell: it does the same calculation without the widgets.

In [ ]:
# Only if clicking Run shows nothing: the same calculation, without the widgets
if "sliders" in globals() and "reps" in globals():
    params = {**BASELINE, **{name: round(s.value, 1) for name, s in sliders.items()}}
    display(summarize(run_many(params, n=reps.value)))
else:
    print("This cell needs a dictionary of sliders named sliders and a slider named reps, as the request asked.")

## Step 6. Write the do-not-use rules (3 minutes)
Three rules, each specific enough to check (double-click the cell below to type). What's missing from this model
that a real unit has?

1.
2.
3.

## Before the debrief, jot down
1. Which proposal would you recommend, on which metric, and with what interval?
2. What would you need from the birth center's own data before trusting this model?
3. One thing this dashboard must refuse to do.

*The unit's structure follows Mark Isken's teaching model of an obstetrics unit (Oakland University).*